# Generate hrtfpykit plots for the SONICOM ecosystem example

This notebook loads the ARI `hrtf b_nh5.sofa` file used in the SONICOM ecosystem datafile example and regenerates the `hrtfpykit` images shown in the repository README. It assumes `hrtfpykit` is already installed in the Python environment running the notebook.

## Imports and paths

The notebook can be run from the repository root or from the `notebooks/` directory.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from hrtfpykit.hrtf import load_hrtf
from hrtfpykit.plots import (
    plot_absolute_ild,
    plot_absolute_itd,
    plot_amplitude,
    plot_ild,
    plot_ild_fd,
    plot_itd,
    plot_elevation_spectrum,
    plot_etc_plane,
    plot_magnitude,
    plot_plane_grid,
    plot_source_grid,
    plot_spectrum_plane,
)

repo_root = Path.cwd()
if not (repo_root / "data" / "hrtf b_nh5.sofa").exists():
    repo_root = repo_root.parent

sofa_path = repo_root / "data" / "hrtf b_nh5.sofa"
comparable_dir = repo_root / "images" / "hrtfpykit" / "comparable"
additional_dir = repo_root / "images" / "hrtfpykit" / "additional"

comparable_dir.mkdir(parents=True, exist_ok=True)
additional_dir.mkdir(parents=True, exist_ok=True)

sofa_path


## Load the SOFA file

`load_hrtf` returns one `HRTF` object with synchronized `IR`, `TF`, and `Sources` views. The ARI file can expose custom metadata fields, so SOFA convention warnings may appear while loading. The warnings do not prevent visualization.

In [ ]:
hrtf = load_hrtf(sofa_path)

print("IR:", None if hrtf.IR.values is None else hrtf.IR.values.shape)
print("TF:", None if hrtf.TF.values is None else hrtf.TF.values.shape)
print("Sources:", hrtf.Sources.get_positions(angle_unit="degrees").shape)


## Helper for saving figures

In [ ]:
def save_figure(fig, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(destination, dpi=450, bbox_inches="tight")
    plt.close(fig)
    return destination


## Plots matching the SONICOM ecosystem views

These figures correspond to the downloaded SONICOM ecosystem images: horizontal-plane ETC for each ear, median-plane spectral magnitude for each ear, absolute ITD over the horizontal plane, and the source grid.

In [ ]:
matching_plots = [
    (
        comparable_dir / "etc-horizontal-left.png",
        lambda: plot_etc_plane(
            hrtf,
            ear="left",
            plane="horizontal",
            azimuth_range_mode="-180-180",
            show=False,
        ),
    ),
    (
        comparable_dir / "etc-horizontal-right.png",
        lambda: plot_etc_plane(
            hrtf,
            ear="right",
            plane="horizontal",
            azimuth_range_mode="-180-180",
            show=False,
        ),
    ),
    (
        comparable_dir / "spectrum-median-left.png",
        lambda: plot_spectrum_plane(
            hrtf,
            ear="left",
            plane="median",
            freq_max=18000.0,
            show=False,
        ),
    ),
    (
        comparable_dir / "spectrum-median-right.png",
        lambda: plot_spectrum_plane(
            hrtf,
            ear="right",
            plane="median",
            freq_max=18000.0,
            show=False,
        ),
    ),
    (
        comparable_dir / "absolute-itd-horizontal.png",
        lambda: plot_absolute_itd(hrtf, plane_angle=0.0, show=False),
    ),
    (
        comparable_dir / "source-grid.png",
        lambda: plot_source_grid(hrtf, show=False),
    ),
]

for destination, make_figure in matching_plots:
    print(save_figure(make_figure(), destination))


## Additional hrtfpykit plots

These figures demonstrate extra visualization options that can be generated from the same loaded `HRTF` object.

In [ ]:
additional_plots = [
    (additional_dir / "amplitude-default.png", lambda: plot_amplitude(hrtf, show=False)),
    (additional_dir / "magnitude-default.png", lambda: plot_magnitude(hrtf, show=False)),
    (additional_dir / "itd-horizontal.png", lambda: plot_itd(hrtf, show=False)),
    (additional_dir / "ild-horizontal.png", lambda: plot_ild(hrtf, show=False)),
    (additional_dir / "ild-frequency-horizontal.png", lambda: plot_ild_fd(hrtf, show=False)),
    (additional_dir / "plane-grid.png", lambda: plot_plane_grid(hrtf, show=False)),
    (additional_dir / "absolute-ild-horizontal.png", lambda: plot_absolute_ild(hrtf, show=False)),
    (
        additional_dir / "elevation-spectrum-default.png",
        lambda: plot_elevation_spectrum(hrtf, freq_max=18000.0, show=False),
    ),
]

for destination, make_figure in additional_plots:
    print(save_figure(make_figure(), destination))
